In [9]:
using Random
using DynamicPolynomials
using SumOfSquares
import CSDP
import SumOfSquares.solve
using JuMP, MosekTools, LinearAlgebra

Example B.4

In [8]:
@polyvar x[1:2] y[1:2] 
p = 2+3*x[1]*y[1]-5*x[2]*y[2]+4*y[1]^2
# we can use a substitution trick to reduce the number of variables and the degree of the polynomial
p_sub = subs(p, x[2] => 1 - x[1], y[2] => 1 - y[1])
println("p_sub = ", p_sub)
Sx = @set x[1] >= 0 && y[1] >= 0 && y[1] <= 1 && x[1] <= 1

p_sub = -3 + 5*y[1] + 5*x[1] + 4*y[1]^2 - 2*x[1]*y[1]


Basic semialgebraic Set defined by no equality
4 inequalities
 x[1] ≥ 0
 y[1] ≥ 0
 1 - y[1] ≥ 0
 1 - x[1] ≥ 0


In [4]:
model = SOSModel(Mosek.Optimizer)
@variable(model, t)
@objective(model, Min, t)
@constraint(model, c, p_sub <= t, domain = Sx, maxdegree = 2)
optimize!(model)
println("Solution: $(value(t))")

v = moment_matrix(model[:c])
nu = atomic_measure(v, 1e-3)
print(nu)

Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 5               
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 1               
  Scalar variables       : 8               
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Eliminator - tries                  : 0                 time                   : 0.00            
Lin. dep.  - tries                  : 0                 time                   : 0.00            
Lin. dep.  - primal attempts        : 0                 successes              : 0               
Lin. dep.  - dual attempts          : 0                 successes              : 0               
Lin. dep.  - primal deps.           : 0                 dual deps.             : 0               
Presol

In [7]:
model = SOSModel(Mosek.Optimizer)
@variable(model, t)
@objective(model, Min, t)
@constraint(model, c, p_sub <= t, domain = Sx, maxdegree = 4)
optimize!(model)
println("Solution: $(value(t))")

v = moment_matrix(model[:c])
nu = atomic_measure(v, 1e-2)
print(nu)

Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 10              
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 1               
  Matrix variables       : 5 (scalarized: 30)
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.00            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0         

In [2]:
# Functions to randomly generate multilinear polynomials (with and without variable repeats in each monomial)
# Polynomials without variable repeats in the same monomial are non-absentminded.

function random_multilinear_poly(varlists; nterms=5, coeff_range=(-5,5), seed=1234)
    Random.seed!(seed)
    allvars = reduce(vcat, varlists)
    n = length(allvars)
    polys = DynamicPolynomials.zero(allvars[1])
    terms = Set{Vector{Int}}()
    while length(terms) < nterms
        exps = [rand(Bool) for _ in 1:n]
        if any(exps)
            push!(terms, exps)
        end
    end
    for exps in terms
        coeff = rand(coeff_range[1]:coeff_range[2])
        term = coeff*prod(allvars[i]^exps[i] for i in 1:n)
        polys += term
    end
    return polys
end

function random_multilinear_poly_no_repeats(varlists; nterms=5, coeff_range=(-5,5), seed=1234)
    Random.seed!(seed)
    nvars = length(varlists)
    polys = DynamicPolynomials.zero(reduce(vcat, varlists)[1])
    terms = Set{Vector{Union{Nothing,Int}}}()
    while length(terms) < nterms
        choices = [rand(0:length(vs)) for vs in varlists]
        if any(choices .> 0)
            push!(terms, choices)
        end
    end
    for choices in terms
        coeff = rand(coeff_range[1]:coeff_range[2])
        if coeff == 0
            continue
        end
        term = coeff
        for (i, idx) in enumerate(choices)
            if idx > 0
                term *= varlists[i][idx]
            end
        end
        polys += term
    end
    return polys
end

random_multilinear_poly_no_repeats (generic function with 1 method)

Example of single-player IREFG $\mathcal{G}_1$, without infoset repeats in variables. This is a NAM game, so we should expect:
1. Ex-ante optima at the vertices
2. Convergence at a finite level of the hierarchy
3. Extraction always possible at the $\ell+1$ level

In [ ]:
@polyvar x[1:2]
@polyvar y[1:2] 
@polyvar z[1:2]
test2 = random_multilinear_poly_no_repeats([x,y,z],nterms=6)

-4z₁ + x₂y₂ + x₂y₂z₁ - 3x₂y₁z₂ - 3x₂y₁z₁

In [ ]:
gs = [x; 1 - sum(x); 1 - sum(x.^2); y; 1 - sum(y); 1 - sum(y.^2); z; 1 - sum(z); 1 - sum(z.^2)]
hs = [sum(x) - 1; sum(y) - 1; sum(z) - 1]
Sg = basic_semialgebraic_set(FullSpace(), gs) 
Sh = algebraic_set(hs)

for var in [x, y, z]
    Sx = intersect(Sx, algebraic_set([i*(1-i) for i in var]))
end

In [ ]:
model = SOSModel(Mosek.Optimizer)
@variable(model, t)
@objective(model, Min, t)
@constraint(model, c, test2 <= t, domain = Sx, maxdegree = 2)
optimize!(model)
println("Solution: $(value(t))")

v = moment_matrix(model[:c])
nu = atomic_measure(v, 0.5e-1)
print(nu)

Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 63              
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 11              
  Matrix variables       : 1 (scalarized: 91)
  Integer variables      : 0               

Optimizer started.
Presolve started.
Eliminator - tries                  : 0                 time                   : 0.00            
Lin. dep.  - tries                  : 0                 time                   : 0.00            
Lin. dep.  - primal attempts        : 0                 successes              : 0               
Lin. dep.  - dual attempts          : 0                 successes              : 0               
Lin. dep.  - primal deps.           : 0                 dual deps.             : 0               
Pres

In the generated example, $\ell+1=4$ so let us try setting degree 4 of the hierarchy.

In [ ]:
model = SOSModel(Mosek.Optimizer)
@variable(model, t)
@objective(model, Min, t)
@constraint(model, c, test2 <= t, domain = Sx, maxdegree = 4)
optimize!(model)
println("Solution: $(value(t))")

v = moment_matrix(model[:c])
nu = atomic_measure(v, 1e-2)
print(nu)

Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 8               
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 1               
  Matrix variables       : 13 (scalarized: 742)
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 2                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.01            
Lin. dep.  - primal attempts        : 1                 successes

Example for a randomly generated single-player IREFG $\mathcal{G}_1$. Note that we do not disallow repeats of infosets, so this game is considered absent-minded. Theoretically, we expect asymptotic convergence to the ex-ante optimum value.

In [3]:
@polyvar x[1:3]
@polyvar y[1:3] 
test1 = random_multilinear_poly([x,y],nterms=5)

4x₁x₃ + 2x₂x₃y₃ - 5x₁x₂y₃ + x₁x₂y₁ - 4x₂x₃y₂y₃

In [4]:
gs = [x; 1 - sum(x); 1 - sum(x.^2); y; 1 - sum(y); 1 - sum(y.^2)]
hs = [sum(x) - 1; sum(y) - 1]
Sg = basic_semialgebraic_set(FullSpace(), gs) 
Sh = algebraic_set(hs)
Sx = intersect(Sh, Sg)

Basic semialgebraic Set defined by 2 equalities
 -1//1 + x[3] + x[2] + x[1] = 0
 -1//1 + y[3] + y[2] + y[1] = 0
10 inequalities
 x[1] ≥ 0
 x[2] ≥ 0
 x[3] ≥ 0
 1//1 - x[3] - x[2] - x[1] ≥ 0
 1//1 - x[3]^2 - x[2]^2 - x[1]^2 ≥ 0
 y[1] ≥ 0
 y[2] ≥ 0
 y[3] ≥ 0
 1//1 - y[3] - y[2] - y[1] ≥ 0
 1//1 - y[3]^2 - y[2]^2 - y[1]^2 ≥ 0


In [5]:

model = SOSModel(Mosek.Optimizer)
@variable(model, t)
@objective(model, Min, t)
@constraint(model, c, test1 <= t, domain = Sx, maxdegree = 2)
optimize!(model)
println("Solution: $(value(t))")

v = moment_matrix(model[:c])
nu = atomic_measure(v, 0.5e-1)
print(nu)

Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 20              
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 11              
  Matrix variables       : 1 (scalarized: 28)
  Integer variables      : 0               

Optimizer started.
Presolve started.
Eliminator - tries                  : 0                 time                   : 0.00            
Lin. dep.  - tries                  : 0                 time                   : 0.00            
Lin. dep.  - primal attempts        : 0                 successes              : 0               
Lin. dep.  - dual attempts          : 0                 successes              : 0               
Lin. dep.  - primal deps.           : 0                 dual deps.             : 0               
Pres

In [6]:
model = SOSModel(Mosek.Optimizer)
@variable(model, t)
@objective(model, Min, t)
@constraint(model, c, test1 <= t, domain = Sx, maxdegree = 4)
optimize!(model)
println("Solution: $(value(t))")

v = moment_matrix(model[:c])
nu = atomic_measure(v, 1e-2)
print(nu)

Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 70              
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 1               
  Matrix variables       : 11 (scalarized: 686)
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.00            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

In [7]:
model = SOSModel(Mosek.Optimizer)
@variable(model, t)
@objective(model, Min, t)
@constraint(model, c, test1 <= t, domain = Sx, maxdegree = 6)
optimize!(model)
println("Solution: $(value(t))")

v = moment_matrix(model[:c])
nu = atomic_measure(v, 1e-2)
print(nu)

Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 210             
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 1               
  Matrix variables       : 11 (scalarized: 7630)
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.00            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0      